In [3]:
import os
import glob
import numpy as np
import nibabel as nib
from collections import defaultdict
import json

# ================= CONFIG =================
# We will scan the HGG folder for a robust statistical sample
DATASET_ROOT = r"/home/23bq1a4293/Personal/project/seg_data/glioma/MICCAI_BraTS_2019_Data_Training/LGG"
MAX_PATIENTS_TO_SCAN = 15  # 15 patients is enough to get perfect statistics

def analyze_dataset(root_dir, max_samples):
    print(f"🔍 Starting Deep Analysis on: {root_dir}")
    
    patient_dirs = [os.path.join(root_dir, d) for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))]
    patient_dirs = sorted(patient_dirs)[:max_samples]
    
    stats = {
        "num_patients_scanned": len(patient_dirs),
        "shapes": defaultdict(int),
        "spacings (resolution)": defaultdict(int),
        "orientations (affines)": [],
        "intensities": {"min": [], "max": [], "mean": []},
        "class_imbalance (voxels)": {0: 0, 1: 0, 2: 0, 4: 0},
        "missing_files": 0
    }
    
    for p_dir in patient_dirs:
        files = glob.glob(os.path.join(p_dir, "*.nii*"))
        
        t1ce = next((f for f in files if "t1ce" in f.lower()), None)
        seg  = next((f for f in files if "seg" in f.lower()), None)
        
        if not t1ce or not seg:
            stats["missing_files"] += 1
            continue
            
        # 1. Spatial Analysis (Shape, Spacing, Affine)
        img = nib.load(t1ce)
        header = img.header
        
        shape_str = str(img.shape)
        stats["shapes"][shape_str] += 1
        
        spacing_str = str(np.round(header.get_zooms()[:3], 3))
        stats["spacings (resolution)"][spacing_str] += 1
        
        if len(stats["orientations (affines)"]) < 3:
            stats["orientations (affines)"].append(np.round(img.affine, 1).tolist())
            
        # 2. Intensity Analysis (Excluding background zeros)
        data = img.get_fdata(dtype=np.float32)
        brain_mask = data > 0
        if np.any(brain_mask):
            stats["intensities"]["min"].append(float(np.min(data[brain_mask])))
            stats["intensities"]["max"].append(float(np.max(data[brain_mask])))
            stats["intensities"]["mean"].append(float(np.mean(data[brain_mask])))
            
        # 3. Class Imbalance Analysis (Ground Truth)
        seg_data = nib.load(seg).get_fdata().astype(np.uint8)
        unique, counts = np.unique(seg_data, return_counts=True)
        for u, c in zip(unique, counts):
            if u in stats["class_imbalance (voxels)"]:
                stats["class_imbalance (voxels)"][u] += int(c)
                
    # Aggregate intensities
    stats["intensities"]["global_min"] = min(stats["intensities"]["min"])
    stats["intensities"]["global_max"] = max(stats["intensities"]["max"])
    stats["intensities"]["average_mean"] = sum(stats["intensities"]["mean"]) / len(stats["intensities"]["mean"])
    del stats["intensities"]["min"], stats["intensities"]["max"], stats["intensities"]["mean"]

    # Calculate Percentages for Class Imbalance
    total_voxels = sum(stats["class_imbalance (voxels)"].values())
    stats["class_percentages"] = {
        "Background (0)": f"{(stats['class_imbalance (voxels)'][0] / total_voxels) * 100:.2f}%",
        "Necrotic/NET (1)": f"{(stats['class_imbalance (voxels)'][1] / total_voxels) * 100:.3f}%",
        "Edema (2)": f"{(stats['class_imbalance (voxels)'][2] / total_voxels) * 100:.3f}%",
        "Enhancing (4)": f"{(stats['class_imbalance (voxels)'][4] / total_voxels) * 100:.3f}%",
    }
    
    print("\n✅ Analysis Complete. Copy and paste this output to me:\n")
    print(json.dumps(stats, indent=4))

# Run it
if os.path.exists(DATASET_ROOT):
    analyze_dataset(DATASET_ROOT, MAX_PATIENTS_TO_SCAN)
else:
    print(f"Folder not found: {DATASET_ROOT}")

🔍 Starting Deep Analysis on: /home/23bq1a4293/Personal/project/seg_data/glioma/MICCAI_BraTS_2019_Data_Training/LGG

✅ Analysis Complete. Copy and paste this output to me:

{
    "num_patients_scanned": 15,
    "shapes": {
        "(240, 240, 155)": 15
    },
    "spacings (resolution)": {
        "[1. 1. 1.]": 15
    },
    "orientations (affines)": [
        [
            [
                -1.0,
                0.0,
                0.0,
                -0.0
            ],
            [
                0.0,
                -1.0,
                0.0,
                239.0
            ],
            [
                0.0,
                0.0,
                1.0,
                0.0
            ],
            [
                0.0,
                0.0,
                0.0,
                1.0
            ]
        ],
        [
            [
                -1.0,
                0.0,
                0.0,
                -0.0
            ],
            [
                0.0,
              

In [3]:
%matplotlib inline
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output

from monai.inferers import sliding_window_inference
from monai.networks.nets import SwinUNETR

# Import directly from your existing dataset script!
from dataset import build_dataset_dicts, get_train_val_transforms

# ================= CONFIG =================
checkpoint   = "/home/23bq1a4293/Personal/project/IMAGE_SEGMENTATION/glioma/swin_best_mean.pth"
PATCH_SIZE  = (128, 128, 128)
OVERLAP     = 0.5
device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

FOLDER_MAP = {
    "/home/23bq1a4293/Personal/project/seg_data/glioma/MICCAI_BraTS_2019_Data_Training/HGG": 0,
    "/home/23bq1a4293/Personal/project/seg_data/glioma/MICCAI_BraTS_2019_Data_Training/LGG": 1,
}

# ================= COLORMAP =================
COLORS = {
    "WT": (0.0, 1.0, 0.0, 0.45),
    "TC": (0.0, 0.4, 1.0, 0.55),
    "ET": (1.0, 0.1, 0.1, 0.70),
}
legend_patches = [
    mpatches.Patch(color=COLORS["WT"][:3], label='Whole Tumor (WT)'),
    mpatches.Patch(color=COLORS["TC"][:3], label='Tumor Core (TC)'),
    mpatches.Patch(color=COLORS["ET"][:3], label='Enhancing Tumor (ET)'),
]

# ================= LOAD SWIN-UNETR =================
model = SwinUNETR(
    in_channels=4, out_channels=3, feature_size=48, use_checkpoint=True
).to(device)
model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
model.eval()
print(f"✅ SwinUNETR loaded")

# ================= MONAI LOADER =================
print("Building dataset dictionaries...")
data_dicts = build_dataset_dicts(FOLDER_MAP)
_, val_transform = get_train_val_transforms()

# Create a mapping by patient ID for the dropdown
patients_dict = {d['t1'].split('/')[-2]: d for d in data_dicts}
available_patients = sorted(list(patients_dict.keys()))

inference_cache = {}

def dice_score(p, g):
    intersection = np.sum(p * g)
    if np.sum(p) + np.sum(g) == 0: return 1.0
    return 2.0 * intersection / (np.sum(p) + np.sum(g))

def run_inference(pid):
    if pid in inference_cache:
        return inference_cache[pid]
    
    # 1. Load and transform exactly like validation
    raw_dict = patients_dict[pid]
    data = val_transform(raw_dict)
    
    # 2. Stack image modalities
    img_tensor = torch.cat([data["t1"], data["t1ce"], data["t2"], data["flair"]], dim=0) 
    img_tensor = img_tensor.unsqueeze(0).to(device) # (1, 4, H, W, D)
    
    # 3. Ground Truth is already processed by ConvertBraTSLabelsd
    gt_multi = data["label"].numpy() # (3, H, W, D)
    
    # 4. Predict
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            logits = sliding_window_inference(
                img_tensor, roi_size=PATCH_SIZE,
                sw_batch_size=1, predictor=model, overlap=OVERLAP
            )
            
    probs = torch.sigmoid(logits).squeeze(0).cpu().numpy()
    pred_np = (probs > 0.5).astype(np.uint8)

    tc_d = dice_score(pred_np[0], gt_multi[0])
    wt_d = dice_score(pred_np[1], gt_multi[1])
    et_d = dice_score(pred_np[2], gt_multi[2])
    per_class = [tc_d, wt_d, et_d]

    # Use T1ce [1] for background
    bg_img = img_tensor[0, 1].cpu().numpy()

    result = {
        "bg":        bg_img,        # (H,W,D)
        "gt":        gt_multi,      # (3,H,W,D)
        "pred":      pred_np,       # (3,H,W,D)
        "dice":      per_class,
        "mean_dice": float(np.mean(per_class)),
        "num_slices": pred_np.shape[3],
        "best_slice": int(np.argmax(gt_multi[1].sum(axis=(0, 1))) if gt_multi[1].sum() > 0 else bg_img.shape[2]//2),
    }
    inference_cache[pid] = result
    return result

# ================= WIDGETS =================
patient_dropdown = widgets.Dropdown(options=available_patients, description="Patient:")
slice_slider = widgets.IntSlider(min=0, max=150, step=1, value=75, description="Z-Slice:")
run_button = widgets.Button(description="▶ Run Inference", button_style="success")
best_slice_button = widgets.Button(description="⚡ Best Slice", button_style="info")
dice_label = widgets.HTML(value="<b>Dice:</b> run inference first")
output = widgets.Output()

# ================= PLOT =================
def create_overlay(mask_multi_sl):
    tc, wt, et = mask_multi_sl[0], mask_multi_sl[1], mask_multi_sl[2]
    H, W = tc.shape
    overlay = np.zeros((H, W, 4), dtype=np.float32)
    for mask_slice, (r, g, b, a) in zip([wt, tc, et], [COLORS["WT"], COLORS["TC"], COLORS["ET"]]):
        where = mask_slice > 0
        overlay[where] = [r, g, b, a]
    return overlay

def plot_current(pid, slice_idx):
    if pid not in inference_cache:
        with output:
            clear_output(wait=True)
            print(f"⏳ Click '▶ Run Inference' to process {pid}")
        return

    r = inference_cache[pid]
    slice_idx = min(slice_idx, r["num_slices"] - 1)
    
    img_sl  = r["bg"][:, :, slice_idx]
    gt_sl   = r["gt"][:, :, :, slice_idx]     
    pred_sl = r["pred"][:, :, :, slice_idx]   

    with output:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 4, figsize=(22, 5))

        axes[0].imshow(img_sl.T, cmap="gray", origin="lower")
        axes[0].set_title(f"T1ce (Slice {slice_idx})", fontsize=11)
        axes[0].axis("off")

        overlay_gt = create_overlay(gt_sl)
        axes[1].imshow(img_sl.T, cmap="gray", origin="lower")
        axes[1].imshow(overlay_gt.transpose(1, 0, 2), origin="lower")
        axes[1].set_title("Ground Truth", fontsize=11)
        axes[1].axis("off")

        overlay_pred = create_overlay(pred_sl)
        axes[2].imshow(img_sl.T,  cmap="gray", origin="lower")
        axes[2].imshow(overlay_pred.transpose(1, 0, 2), origin="lower")
        axes[2].set_title("Prediction", fontsize=11)
        axes[2].axis("off")

        diff = np.zeros((*img_sl.shape, 4), dtype=np.float32)
        fp = ((pred_sl[1] - gt_sl[1]) == 1)   
        fn = ((gt_sl[1] - pred_sl[1]) == 1)   
        diff[fp] = [1, 0, 0, 0.7]   
        diff[fn] = [0, 0, 1, 0.7]   
        
        axes[3].imshow(img_sl.T, cmap="gray", origin="lower")
        axes[3].imshow(diff.transpose(1, 0, 2), origin="lower")
        axes[3].set_title("Error Map (Red=FP, Blue=FN)", fontsize=11)
        axes[3].axis("off")

        fig.legend(handles=legend_patches, loc="lower center", ncol=3, fontsize=10, bbox_to_anchor=(0.5, -0.05))
        d = r["dice"]
        plt.suptitle(f"{pid} | Slice {slice_idx} | Mean Dice: {r['mean_dice']:.4f} | TC: {d[0]:.3f} WT: {d[1]:.3f} ET: {d[2]:.3f}", fontsize=12, fontweight="bold")
        plt.tight_layout()
        plt.show()

# ================= CALLBACKS =================
def on_run_clicked(b):
    pid = patient_dropdown.value
    run_button.description = "⏳ Running..."
    run_button.disabled = True
    
    result = run_inference(pid)
    slice_slider.max = result["num_slices"] - 1
    slice_slider.value = result["best_slice"]
    
    d = result["dice"]
    dice_label.value = f"<b>Mean Dice: {result['mean_dice']:.4f}</b> | TC: {d[0]:.3f} | WT: {d[1]:.3f} | ET: {d[2]:.3f}"
    
    run_button.description = "▶ Run Inference"
    run_button.disabled = False
    plot_current(pid, slice_slider.value)

def on_best_slice(b):
    pid = patient_dropdown.value
    if pid in inference_cache:
        slice_slider.value = inference_cache[pid]["best_slice"]

def on_slice_change(change):
    plot_current(patient_dropdown.value, change["new"])

run_button.on_click(on_run_clicked)
best_slice_button.on_click(on_best_slice)
slice_slider.observe(on_slice_change, names="value")

controls = widgets.VBox([
    widgets.HBox([patient_dropdown, run_button, best_slice_button]),
    slice_slider,
    dice_label,
])
display(controls, output)

✅ SwinUNETR loaded
Building dataset dictionaries...
[Dataset] Found 335 valid patients.


monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.


Output()

In [1]:
%matplotlib inline
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output

from monai.inferers import sliding_window_inference
from monai.networks.nets import SwinUNETR, SegResNet

# Import directly from your existing dataset script!
from dataset import build_dataset_dicts, get_train_val_transforms

# ================= CONFIG =================
# Update these paths if necessary
SWIN_CKPT       = "swin_best_mean.pth"
SEGRESNET_CKPT  = "segresnet_best_mean.pth"

PATCH_SIZE  = (128, 128, 128)
OVERLAP     = 0.5
device      = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

FOLDER_MAP = {
    "/home/23bq1a4293/Personal/project/seg_data/glioma/MICCAI_BraTS_2019_Data_Training/HGG": 0,
    "/home/23bq1a4293/Personal/project/seg_data/glioma/MICCAI_BraTS_2019_Data_Training/LGG": 1,
}

# ================= COLORMAP =================
COLORS = {
    "WT": (0.0, 1.0, 0.0, 0.45),
    "TC": (0.0, 0.4, 1.0, 0.55),
    "ET": (1.0, 0.1, 0.1, 0.70),
}
legend_patches = [
    mpatches.Patch(color=COLORS["WT"][:3], label='Whole Tumor (WT)'),
    mpatches.Patch(color=COLORS["TC"][:3], label='Tumor Core (TC)'),
    mpatches.Patch(color=COLORS["ET"][:3], label='Enhancing Tumor (ET)'),
]

# ================= LOAD MODELS =================
print("Loading SwinUNETR (Global Geometry)...")
swin = SwinUNETR(
    in_channels=4, out_channels=3, feature_size=48, use_checkpoint=True
).to(device)
swin.load_state_dict(torch.load(SWIN_CKPT, map_location=device, weights_only=True))
swin.eval()

print("Loading SegResNet (Local Textures)...")
segresnet = SegResNet(
    blocks_down=[1, 2, 2, 4],
    blocks_up=[1, 1, 1],
    init_filters=16,
    in_channels=4,
    out_channels=3,
    dropout_prob=0.2,
).to(device)
segresnet.load_state_dict(torch.load(SEGRESNET_CKPT, map_location=device, weights_only=True))
segresnet.eval()
print(f"✅ Both Models Loaded!")

# ================= MONAI LOADER =================
print("Building dataset dictionaries...")
data_dicts = build_dataset_dicts(FOLDER_MAP)
_, val_transform = get_train_val_transforms()

# Create a mapping by patient ID for the dropdown
patients_dict = {d['t1'].split('/')[-2]: d for d in data_dicts}
available_patients = sorted(list(patients_dict.keys()))

inference_cache = {}

def dice_score(p, g):
    intersection = np.sum(p * g)
    if np.sum(p) + np.sum(g) == 0: return 1.0
    return 2.0 * intersection / (np.sum(p) + np.sum(g))

def run_inference(pid):
    if pid in inference_cache:
        return inference_cache[pid]
    
    # 1. Load and transform exactly like validation
    raw_dict = patients_dict[pid]
    data = val_transform(raw_dict)
    
    # 2. Stack image modalities
    img_tensor = torch.cat([data["t1"], data["t1ce"], data["t2"], data["flair"]], dim=0) 
    img_tensor = img_tensor.unsqueeze(0).to(device) # (1, 4, H, W, D)
    
    # 3. Ground Truth is already processed by ConvertBraTSLabelsd
    gt_multi = data["label"].numpy() # (3, H, W, D)
    
    # 4. Predict using Ensemble
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            # SwinUNETR Forward
            swin_logits = sliding_window_inference(
                img_tensor, roi_size=PATCH_SIZE,
                sw_batch_size=1, predictor=swin, overlap=OVERLAP
            )
            # SegResNet Forward
            seg_logits = sliding_window_inference(
                img_tensor, roi_size=PATCH_SIZE,
                sw_batch_size=1, predictor=segresnet, overlap=OVERLAP
            )
            
            # Convert to probabilities and average
            swin_probs = torch.sigmoid(swin_logits)
            seg_probs = torch.sigmoid(seg_logits)
            ensemble_probs = (swin_probs + seg_probs) / 2.0
            
    probs = ensemble_probs.squeeze(0).cpu().numpy()
    pred_np = (probs > 0.5).astype(np.uint8)

    tc_d = dice_score(pred_np[0], gt_multi[0])
    wt_d = dice_score(pred_np[1], gt_multi[1])
    et_d = dice_score(pred_np[2], gt_multi[2])
    per_class = [tc_d, wt_d, et_d]

    # Use T1ce [1] for background
    bg_img = img_tensor[0, 1].cpu().numpy()

    result = {
        "bg":        bg_img,        # (H,W,D)
        "gt":        gt_multi,      # (3,H,W,D)
        "pred":      pred_np,       # (3,H,W,D)
        "dice":      per_class,
        "mean_dice": float(np.mean(per_class)),
        "num_slices": pred_np.shape[3],
        "best_slice": int(np.argmax(gt_multi[1].sum(axis=(0, 1))) if gt_multi[1].sum() > 0 else bg_img.shape[2]//2),
    }
    inference_cache[pid] = result
    return result

# ================= WIDGETS =================
patient_dropdown = widgets.Dropdown(options=available_patients, description="Patient:")
slice_slider = widgets.IntSlider(min=0, max=150, step=1, value=75, description="Z-Slice:")
run_button = widgets.Button(description="▶ Run Ensemble", button_style="success")
best_slice_button = widgets.Button(description="⚡ Best Slice", button_style="info")
dice_label = widgets.HTML(value="<b>Dice:</b> run inference first")
output = widgets.Output()

# ================= PLOT =================
def create_overlay(mask_multi_sl):
    tc, wt, et = mask_multi_sl[0], mask_multi_sl[1], mask_multi_sl[2]
    H, W = tc.shape
    overlay = np.zeros((H, W, 4), dtype=np.float32)
    for mask_slice, (r, g, b, a) in zip([wt, tc, et], [COLORS["WT"], COLORS["TC"], COLORS["ET"]]):
        where = mask_slice > 0
        overlay[where] = [r, g, b, a]
    return overlay

def plot_current(pid, slice_idx):
    if pid not in inference_cache:
        with output:
            clear_output(wait=True)
            print(f"⏳ Click '▶ Run Ensemble' to process {pid}")
        return

    r = inference_cache[pid]
    slice_idx = min(slice_idx, r["num_slices"] - 1)
    
    img_sl  = r["bg"][:, :, slice_idx]
    gt_sl   = r["gt"][:, :, :, slice_idx]     
    pred_sl = r["pred"][:, :, :, slice_idx]   

    with output:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 4, figsize=(22, 5))

        axes[0].imshow(img_sl.T, cmap="gray", origin="lower")
        axes[0].set_title(f"T1ce (Slice {slice_idx})", fontsize=11)
        axes[0].axis("off")

        overlay_gt = create_overlay(gt_sl)
        axes[1].imshow(img_sl.T, cmap="gray", origin="lower")
        axes[1].imshow(overlay_gt.transpose(1, 0, 2), origin="lower")
        axes[1].set_title("Ground Truth", fontsize=11)
        axes[1].axis("off")

        overlay_pred = create_overlay(pred_sl)
        axes[2].imshow(img_sl.T,  cmap="gray", origin="lower")
        axes[2].imshow(overlay_pred.transpose(1, 0, 2), origin="lower")
        axes[2].set_title("Ensemble Prediction", fontsize=11)
        axes[2].axis("off")

        diff = np.zeros((*img_sl.shape, 4), dtype=np.float32)
        fp = ((pred_sl[1] - gt_sl[1]) == 1)   
        fn = ((gt_sl[1] - pred_sl[1]) == 1)   
        diff[fp] = [1, 0, 0, 0.7]   
        diff[fn] = [0, 0, 1, 0.7]   
        
        axes[3].imshow(img_sl.T, cmap="gray", origin="lower")
        axes[3].imshow(diff.transpose(1, 0, 2), origin="lower")
        axes[3].set_title("Error Map (Red=FP, Blue=FN)", fontsize=11)
        axes[3].axis("off")

        fig.legend(handles=legend_patches, loc="lower center", ncol=3, fontsize=10, bbox_to_anchor=(0.5, -0.05))
        d = r["dice"]
        plt.suptitle(f"{pid} | Slice {slice_idx} | Ensemble Mean Dice: {r['mean_dice']:.4f} | TC: {d[0]:.3f} WT: {d[1]:.3f} ET: {d[2]:.3f}", fontsize=12, fontweight="bold")
        plt.tight_layout()
        plt.show()

# ================= CALLBACKS =================
def on_run_clicked(b):
    pid = patient_dropdown.value
    run_button.description = "⏳ Running..."
    run_button.disabled = True
    
    result = run_inference(pid)
    slice_slider.max = result["num_slices"] - 1
    slice_slider.value = result["best_slice"]
    
    d = result["dice"]
    dice_label.value = f"<b>Ensemble Mean Dice: {result['mean_dice']:.4f}</b> | TC: {d[0]:.3f} | WT: {d[1]:.3f} | ET: {d[2]:.3f}"
    
    run_button.description = "▶ Run Ensemble"
    run_button.disabled = False
    plot_current(pid, slice_slider.value)

def on_best_slice(b):
    pid = patient_dropdown.value
    if pid in inference_cache:
        slice_slider.value = inference_cache[pid]["best_slice"]

def on_slice_change(change):
    plot_current(patient_dropdown.value, change["new"])

run_button.on_click(on_run_clicked)
best_slice_button.on_click(on_best_slice)
slice_slider.observe(on_slice_change, names="value")

controls = widgets.VBox([
    widgets.HBox([patient_dropdown, run_button, best_slice_button]),
    slice_slider,
    dice_label,
])
display(controls, output)

/usr/local/pythonEnv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.2.3) or chardet (7.1.0)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(
I0000 00:00:1789533054.755881 1564980 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789533054.809948 1564980 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789533055.884706 1564980 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different 

Loading SwinUNETR (Global Geometry)...
Loading SegResNet (Local Textures)...
✅ Both Models Loaded!
Building dataset dictionaries...
[Dataset] Found 335 valid patients.


/usr/local/pythonEnv/lib/python3.12/site-packages/monai/utils/deprecate_utils.py:321: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


Output()

In [9]:
import torch
import numpy as np
from tqdm import tqdm
from monai.inferers import sliding_window_inference
from monai.networks.nets import SwinUNETR, SegResNet
from torch.utils.data import DataLoader

# Import directly from your existing dataset script
from dataset import build_dataset_dicts, get_train_val_transforms, MultiTaskBraTSDataset

# ================= CONFIG =================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PATCH_SIZE = (128, 128, 128)
OVERLAP = 0.5

# TODO: Update these paths if necessary
SWIN_CKPT = "swin_best_mean.pth"
SEGRESNET_CKPT = "segresnet_best_mean.pth"

FOLDER_MAP = {
    "/home/23bq1a4293/Personal/project/seg_data/glioma/MICCAI_BraTS_2019_Data_Training/HGG": 0,
    "/home/23bq1a4293/Personal/project/seg_data/glioma/MICCAI_BraTS_2019_Data_Training/LGG": 1,
}

# ================= MODELS =================
print("Loading Models...")
swin = SwinUNETR(in_channels=4, out_channels=3, feature_size=48, use_checkpoint=True).to(device)
swin.load_state_dict(torch.load(SWIN_CKPT, map_location=device, weights_only=True))
swin.eval()

segresnet = SegResNet(
    blocks_down=[1, 2, 2, 4], blocks_up=[1, 1, 1], init_filters=16, 
    in_channels=4, out_channels=3, dropout_prob=0.2
).to(device)
segresnet.load_state_dict(torch.load(SEGRESNET_CKPT, map_location=device, weights_only=True))
segresnet.eval()

# ================= DATASET =================
data_dicts = build_dataset_dicts(FOLDER_MAP)

# Recreate the exact same validation split you used during training
train_size = int(0.8 * len(data_dicts))
val_dicts = data_dicts[train_size:] 

_, val_transform = get_train_val_transforms()
val_ds = MultiTaskBraTSDataset(val_dicts, transform=val_transform)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2)

def dice_score(p, g):
    intersection = np.sum(p * g)
    if np.sum(p) + np.sum(g) == 0: return 1.0
    return 2.0 * intersection / (np.sum(p) + np.sum(g))

tc_scores, wt_scores, et_scores = [], [], []

print(f"\nEvaluating Ensemble on all {len(val_ds)} validation patients...")
with torch.no_grad():
    for img, seg, cls in tqdm(val_loader, desc="Ensemble Inference"):
        img = img.to(device)
        seg = seg.numpy()[0] # Ground truth [3, H, W, D]
        
        with torch.amp.autocast('cuda'):
            # Run both
            swin_logits = sliding_window_inference(img, roi_size=PATCH_SIZE, sw_batch_size=1, predictor=swin, overlap=OVERLAP)
            seg_logits = sliding_window_inference(img, roi_size=PATCH_SIZE, sw_batch_size=1, predictor=segresnet, overlap=OVERLAP)
            
            # Simple 50/50 Average
            ensemble_probs = (torch.sigmoid(swin_logits) + torch.sigmoid(seg_logits)) / 2.0
            
        pred = (ensemble_probs.squeeze(0).cpu().numpy() > 0.5).astype(np.uint8)
        
        # ==============================================================
        # POST-PROCESSING: THE BRATS VOLUME THRESHOLD
        # If the network predicts less than 300 pixels of Enhancing Tumor, 
        # it is highly likely a false positive hallucination on an LGG patient.
        # Set it to 0 to preserve the True Negative (1.0) Dice Score!
        # ==============================================================
        if np.sum(pred[2]) < 300:
            pred[2] = 0
            
        # Calculate metric for this patient
        tc_scores.append(dice_score(pred[0], seg[0]))
        wt_scores.append(dice_score(pred[1], seg[1]))
        et_scores.append(dice_score(pred[2], seg[2]))

# Calculate Final Averages
tc_mean = np.mean(tc_scores)
wt_mean = np.mean(wt_scores)
et_mean = np.mean(et_scores)
overall_mean = np.mean([tc_mean, wt_mean, et_mean])

print("\n" + "="*50)
print(f"FINAL ENSEMBLE VALIDATION SCORES (67 Patients)")
print("="*50)
print(f"Tumor Core (TC):      {tc_mean:.4f}")
print(f"Whole Tumor (WT):     {wt_mean:.4f}")
print(f"Enhancing Tumor (ET): {et_mean:.4f}")
print("-" * 50)
print(f"ENSEMBLE MEAN DICE:   {overall_mean:.4f}")
print("="*50)

Loading Models...
[Dataset] Found 335 valid patients.

Evaluating Ensemble on all 67 validation patients...


Ensemble Inference: 100%|██████████| 67/67 [01:26<00:00,  1.30s/it]


FINAL ENSEMBLE VALIDATION SCORES (67 Patients)
Tumor Core (TC):      0.6679
Whole Tumor (WT):     0.8981
Enhancing Tumor (ET): 0.5809
--------------------------------------------------
ENSEMBLE MEAN DICE:   0.7157


In [2]:
%matplotlib inline
import os
import glob
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output

from monai.inferers import sliding_window_inference
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd,
    NormalizeIntensityd, ToTensord
)

# ================= CONFIG =================
EXTERNAL_PATIENT_DIR = "/home/23bq1a4293/Personal/project/Patients"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PATCH_SIZE = (128, 128, 128)
OVERLAP = 0.5

COLORS = {
    "WT": (0.0, 1.0, 0.0, 0.45),
    "TC": (0.0, 0.4, 1.0, 0.55),
    "ET": (1.0, 0.1, 0.1, 0.70),
}
legend_patches = [
    mpatches.Patch(color=COLORS["WT"][:3], label='Whole Tumor (WT)'),
    mpatches.Patch(color=COLORS["TC"][:3], label='Tumor Core (TC)'),
    mpatches.Patch(color=COLORS["ET"][:3], label='Enhancing Tumor (ET)'),
]

def dice_score(p, g):
    intersection = np.sum(p * g)
    if np.sum(p) + np.sum(g) == 0: return 1.0
    return 2.0 * intersection / (np.sum(p) + np.sum(g))

# ================= PREPROCESSING =================
def load_external_patient(patient_dir):
    # Filter out .zip files
    files = [f for f in glob.glob(os.path.join(patient_dir, "*.nii*")) if not f.endswith(".zip")]
    
    def pick(kw, ex=None):
        return next((f for f in files if kw in os.path.basename(f).lower() and (ex is None or ex not in os.path.basename(f).lower())), None)
    
    t1, t1ce = pick("t1", "t1ce"), pick("t1ce")
    t2, flair = pick("t2"), pick("flair")
    seg = pick("seg") # Try to find ground truth
    
    if not all([t1, t1ce, t2, flair]):
        raise ValueError(f"Could not find all 4 modalities in {patient_dir}.")

    has_gt = seg is not None
    print(f"Loaded Modalities from {os.path.basename(patient_dir)}... (Ground Truth Found: {has_gt})")
    
    data = {"t1": t1, "t1ce": t1ce, "t2": t2, "flair": flair}
    keys_img = ["t1", "t1ce", "t2", "flair"]
    keys_seg = ["seg"] if has_gt else []
    if has_gt:
        data["seg"] = seg

    transform = Compose([
        LoadImaged(keys=keys_img + keys_seg),
        EnsureChannelFirstd(keys=keys_img + keys_seg, channel_dim="no_channel"),
        Orientationd(keys=keys_img + keys_seg, axcodes="RAS"),
        NormalizeIntensityd(keys=keys_img, nonzero=True, channel_wise=True), # Only normalize MRI, not Seg
        ToTensord(keys=keys_img + keys_seg)
    ])

    data = transform(data)
    img_tensor = torch.cat([data["t1"], data["t1ce"], data["t2"], data["flair"]], dim=0).unsqueeze(0).to(device)
    bg_img = img_tensor[0, 1].cpu().numpy()
    
    gt_multi = None
    if has_gt:
        gt_raw = data["seg"].squeeze(0).cpu().numpy() # [H, W, D]
        gt_tc = np.logical_or(gt_raw == 1, gt_raw == 4)
        gt_wt = np.logical_or(np.logical_or(gt_raw == 1, gt_raw == 2), gt_raw == 4)
        gt_et = (gt_raw == 4)
        gt_multi = np.stack([gt_tc, gt_wt, gt_et], axis=0).astype(np.uint8)

    return img_tensor, bg_img, gt_multi

# ================= INFERENCE =================
print("Processing Patient...")
img_tensor, bg_img, gt_multi = load_external_patient(EXTERNAL_PATIENT_DIR)

with torch.no_grad():
    with torch.amp.autocast('cuda'):
        # Note: 'swin' and 'segresnet' models must still be in memory from the previous cell!
        swin_logits = sliding_window_inference(
            img_tensor, roi_size=PATCH_SIZE, sw_batch_size=1, predictor=swin, overlap=OVERLAP
        )
        seg_logits = sliding_window_inference(
            img_tensor, roi_size=PATCH_SIZE, sw_batch_size=1, predictor=segresnet, overlap=OVERLAP
        )
        
        ensemble_probs = (torch.sigmoid(swin_logits) + torch.sigmoid(seg_logits)) / 2.0
        pred_np = (ensemble_probs.squeeze(0).cpu().numpy() > 0.5).astype(np.uint8)

# Compute Dice if GT exists
d_scores, mean_dice = None, None
if gt_multi is not None:
    tc_d = dice_score(pred_np[0], gt_multi[0])
    wt_d = dice_score(pred_np[1], gt_multi[1])
    et_d = dice_score(pred_np[2], gt_multi[2])
    d_scores = [tc_d, wt_d, et_d]
    mean_dice = np.mean(d_scores)

num_slices = pred_np.shape[3]
tumor_slices = np.where(pred_np[1].sum(axis=(0,1)) > 0)[0]
best_slice = int(tumor_slices[len(tumor_slices)//2]) if len(tumor_slices) > 0 else num_slices // 2

# ================= VISUALIZATION =================
def create_overlay(mask_multi_sl):
    tc, wt, et = mask_multi_sl[0], mask_multi_sl[1], mask_multi_sl[2]
    H, W = tc.shape
    overlay = np.zeros((H, W, 4), dtype=np.float32)
    for mask_slice, (r, g, b, a) in zip([wt, tc, et], [COLORS["WT"], COLORS["TC"], COLORS["ET"]]):
        where = mask_slice > 0
        overlay[where] = [r, g, b, a]
    return overlay

slice_slider = widgets.IntSlider(min=0, max=num_slices-1, step=1, value=best_slice, description="Z-Slice:")
output = widgets.Output()

def update_plot(slice_idx):
    img_sl  = bg_img[:, :, slice_idx]
    pred_sl = pred_np[:, :, :, slice_idx]   
    
    with output:
        clear_output(wait=True)
        
        if gt_multi is not None:
            gt_sl = gt_multi[:, :, :, slice_idx]
            fig, axes = plt.subplots(1, 4, figsize=(22, 5))
            
            axes[0].imshow(img_sl.T, cmap="gray", origin="lower")
            axes[0].set_title(f"T1ce (Slice {slice_idx})", fontsize=11)
            axes[0].axis("off")

            axes[1].imshow(img_sl.T, cmap="gray", origin="lower")
            axes[1].imshow(create_overlay(gt_sl).transpose(1, 0, 2), origin="lower")
            axes[1].set_title("Ground Truth", fontsize=11)
            axes[1].axis("off")

            axes[2].imshow(img_sl.T, cmap="gray", origin="lower")
            axes[2].imshow(create_overlay(pred_sl).transpose(1, 0, 2), origin="lower")
            axes[2].set_title("Ensemble Prediction", fontsize=11)
            axes[2].axis("off")
            
            diff = np.zeros((*img_sl.shape, 4), dtype=np.float32)
            diff[(pred_sl[1] - gt_sl[1]) == 1] = [1, 0, 0, 0.7]   # FP
            diff[(gt_sl[1] - pred_sl[1]) == 1] = [0, 0, 1, 0.7]   # FN
            axes[3].imshow(img_sl.T, cmap="gray", origin="lower")
            axes[3].imshow(diff.transpose(1, 0, 2), origin="lower")
            axes[3].set_title("Error Map (Red=FP, Blue=FN)", fontsize=11)
            axes[3].axis("off")
            
            plt.suptitle(f"External Patient | Slice {slice_idx} | Mean Dice: {mean_dice:.4f} | TC: {d_scores[0]:.3f} WT: {d_scores[1]:.3f} ET: {d_scores[2]:.3f}", fontsize=12, fontweight="bold")
        else:
            fig, axes = plt.subplots(1, 2, figsize=(12, 6))
            axes[0].imshow(img_sl.T, cmap="gray", origin="lower")
            axes[0].set_title(f"Original T1ce (Slice {slice_idx})", fontsize=14)
            axes[0].axis("off")

            axes[1].imshow(img_sl.T, cmap="gray", origin="lower")
            axes[1].imshow(create_overlay(pred_sl).transpose(1, 0, 2), origin="lower")
            axes[1].set_title("Ensemble Prediction", fontsize=14)
            axes[1].axis("off")

        fig.legend(handles=legend_patches, loc="lower center", ncol=3, fontsize=12, bbox_to_anchor=(0.5, 0.05 if gt_multi is None else -0.05))
        plt.tight_layout()
        plt.show()

def on_slice_change(change):
    update_plot(change["new"])

slice_slider.observe(on_slice_change, names="value")
display(slice_slider, output)

update_plot(slice_slider.value)

Processing Patient...
Loaded Modalities from Patients... (Ground Truth Found: True)


IntSlider(value=72, description='Z-Slice:', max=154)

Output()

In [1]:
import torch
import numpy as np
from tqdm import tqdm
from monai.inferers import sliding_window_inference
from monai.networks.nets import SwinUNETR, SegResNet
from torch.utils.data import DataLoader

# Import directly from your existing dataset script
from dataset import build_dataset_dicts, get_train_val_transforms, MultiTaskBraTSDataset

# ================= CONFIG =================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PATCH_SIZE = (128, 128, 128)
OVERLAP = 0.5

# TODO: Update these paths if necessary
SWIN_CKPT = "swin_best_mean.pth"
SEGRESNET_CKPT = "segresnet_best_mean.pth"

FOLDER_MAP = {
    "/home/23bq1a4293/Personal/project/seg_data/glioma/MICCAI_BraTS_2019_Data_Training/HGG": 0,
    "/home/23bq1a4293/Personal/project/seg_data/glioma/MICCAI_BraTS_2019_Data_Training/LGG": 1,
}

# ================= MODELS =================
print("Loading Models...")
swin = SwinUNETR(in_channels=4, out_channels=3, feature_size=48, use_checkpoint=True).to(device)
swin.load_state_dict(torch.load(SWIN_CKPT, map_location=device, weights_only=True))
swin.eval()

segresnet = SegResNet(
    blocks_down=[1, 2, 2, 4], blocks_up=[1, 1, 1], init_filters=16, 
    in_channels=4, out_channels=3, dropout_prob=0.2
).to(device)
segresnet.load_state_dict(torch.load(SEGRESNET_CKPT, map_location=device, weights_only=True))
segresnet.eval()

# ================= DATASET =================
# This automatically parses all 335 patients from HGG and LGG
data_dicts = build_dataset_dicts(FOLDER_MAP)

# FIX: We completely remove the train_size split! 
# We pass ALL 335 data_dicts directly into the Validation transform.
_, val_transform = get_train_val_transforms()
val_ds = MultiTaskBraTSDataset(data_dicts, transform=val_transform)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2)

def dice_score(p, g):
    intersection = np.sum(p * g)
    if np.sum(p) + np.sum(g) == 0: return 1.0
    return 2.0 * intersection / (np.sum(p) + np.sum(g))

tc_scores, wt_scores, et_scores = [], [], []

# It should now say "Evaluating Ensemble on all 335 validation patients..."
print(f"\nEvaluating Ensemble on all {len(val_ds)} patients...")
with torch.no_grad():
    for img, seg, cls in tqdm(val_loader, desc="Ensemble Inference"):
        img = img.to(device)
        seg = seg.numpy()[0] # Ground truth [3, H, W, D]
        
        with torch.amp.autocast('cuda'):
            # Run both
            swin_logits = sliding_window_inference(img, roi_size=PATCH_SIZE, sw_batch_size=1, predictor=swin, overlap=OVERLAP)
            seg_logits = sliding_window_inference(img, roi_size=PATCH_SIZE, sw_batch_size=1, predictor=segresnet, overlap=OVERLAP)
            
            # Simple 50/50 Average
            ensemble_probs = (torch.sigmoid(swin_logits) + torch.sigmoid(seg_logits)) / 2.0
            
        pred = (ensemble_probs.squeeze(0).cpu().numpy() > 0.5).astype(np.uint8)
        
        # ==============================================================
        # POST-PROCESSING: THE BRATS VOLUME THRESHOLD
        # ==============================================================
        if np.sum(pred[2]) < 300:
            pred[2] = 0
            
        # Calculate metric for this patient
        tc_scores.append(dice_score(pred[0], seg[0]))
        wt_scores.append(dice_score(pred[1], seg[1]))
        et_scores.append(dice_score(pred[2], seg[2]))

# Calculate Final Averages
tc_mean = np.mean(tc_scores)
wt_mean = np.mean(wt_scores)
et_mean = np.mean(et_scores)
overall_mean = np.mean([tc_mean, wt_mean, et_mean])

print("\n" + "="*50)
print(f"FINAL ENSEMBLE SCORES ({len(val_ds)} Patients)")
print("="*50)
print(f"Tumor Core (TC):      {tc_mean:.4f}")
print(f"Whole Tumor (WT):     {wt_mean:.4f}")
print(f"Enhancing Tumor (ET): {et_mean:.4f}")
print("-" * 50)
print(f"ENSEMBLE MEAN DICE:   {overall_mean:.4f}")
print("="*50)

/usr/local/pythonEnv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.2.3) or chardet (7.1.0)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.3.1+cu118).
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.3.1+cu118).
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.3.1+cu118).
I0000 00:00:1785140264.537800  141058 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1785140264.616409  141058 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructi

Loading Models...


/usr/local/pythonEnv/lib/python3.12/site-packages/monai/utils/deprecate_utils.py:321: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


[Dataset] Found 335 valid patients.

Evaluating Ensemble on all 335 patients...


Ensemble Inference: 100%|██████████| 335/335 [06:32<00:00,  1.17s/it]


FINAL ENSEMBLE SCORES (335 Patients)
Tumor Core (TC):      0.8339
Whole Tumor (WT):     0.9146
Enhancing Tumor (ET): 0.7912
--------------------------------------------------
ENSEMBLE MEAN DICE:   0.8466
